# Image Captioning Model Testing

- This notebook containes a variety of Image Captioning Models tested on inference speed & caption quality to determine which Model is appropriate for captioning Million scale Images.

### Dataloader used to efficiently load image Batches for inference

In [1]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import numpy as np
import torch
import os

# ---------------- Dataset ----------------
class ImageDataset(Dataset):
    def __init__(self, image_dir: str, extensions=(".jpg", ".jpeg", ".png"), as_tensor=False, transform=None):
        self.image_paths = [
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith(extensions)
        ]
        self.as_tensor = as_tensor
        self.transform = transform or (T.ToTensor() if as_tensor else None)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")

        if self.as_tensor and self.transform:
            img = self.transform(img)

        return img, path

# ---------------- Collate Function ----------------
def collate_images(batch):
    imgs, paths = zip(*batch)
    return list(imgs), list(paths)

# ---------------- Prefetching Loader ----------------
class PrefetchLoader:
    """Wrap a DataLoader to prefetch the next batch to GPU asynchronously."""
    def __init__(self, loader, device, as_tensor=False):
        self.loader = iter(loader)
        self.device = device
        self.as_tensor = as_tensor
        self.stream = torch.cuda.Stream() if device == "cuda" else None
        self.next_batch = None
        self._prefetch()

    def _prefetch(self):
        try:
            imgs, paths = next(self.loader)
        except StopIteration:
            self.next_batch = None
            return

        # Prefetching logic
        if self.device == "cuda":
            with torch.cuda.stream(self.stream):
                if self.as_tensor:
                    # Move tensors to GPU asynchronously
                    imgs = [img.to(self.device, non_blocking=True) for img in imgs]
                # else: keep PIL images on CPU (LLaVA/CLIP-style models expect CPU PILs)
        else:
            if self.as_tensor:
                imgs = [img.to(self.device) for img in imgs]

        self.next_batch = (imgs, paths)

    def __iter__(self):
        return self

    def __next__(self):
        if self.next_batch is None:
            raise StopIteration
        batch = self.next_batch
        self._prefetch()
        return batch

# ---------------- DataLoader setup ----------------
def get_image_loader(image_dir: str, batch_size=4, num_workers=4, device='cpu', as_tensor=False):
    dataset = ImageDataset(image_dir, as_tensor=as_tensor)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
        collate_fn=collate_images,
        pin_memory=True
    )
    prefetch_loader = PrefetchLoader(loader, device=device, as_tensor=as_tensor)
    return prefetch_loader, len(dataset)

## Model 1 - llava-hf/llava-1.5-7b-hf

- This is the model used in the ["Understanding Bias in Large-Scale Visual Datasets"](https://arxiv.org/pdf/2412.01876v1) paper, using 4bit quantization as per the  [github repo](https://github.com/boyazeng/understand_bias/blob/main/transformations/caption/transform.py).

- **NOT VIABLE**: Model takes 27.38 seconds to process 25 images using an batch_size=2 (larger batch sizes slow down inference due to model scale) even when scaled down with the 4bit quantization

In [2]:
import torch
from PIL import Image
import time
from transformers import LlavaProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F

# Load the dataset
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
prefetch_loader, dataset_size = get_image_loader(image_dir, batch_size=2, num_workers=0, device="cuda")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "llava-hf/llava-1.5-7b-hf"
processor = LlavaProcessor.from_pretrained(model_id, use_fast=True)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="cuda"
)

def generate_captions_batch(images, paths, caption_type="short"):
    if caption_type == "short":
        prompt = "USER: <image>\nDescribe this image in one sentence.\nASSISTANT:"
        max_tokens = 50
    else:
        prompt = "USER: <image>\nDescribe this image in one paragraph.\nASSISTANT:"
        max_tokens = 150

    # Combine prompts with each image
    prompts = [prompt] * len(images)

    # Prepare inputs — LLaVA can handle batched images
    inputs = processor(
        images=images,
        text=prompts,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens)
    print(f"Time taken: {time.time() - start_time}")

    # Decode each caption separately
    captions = []
    for output in outputs:
        text = processor.decode(output, skip_special_tokens=True)
        text = text.split("ASSISTANT:")[-1].strip()
        captions.append(text)

    return list(zip(paths, captions))

all_results = []

for imgs, paths in prefetch_loader:
    # Generate captions for the whole batch
    batch_results = generate_captions_batch(imgs, paths, caption_type="short")
        
    all_results.extend(batch_results)

# Example output
for path, caption in all_results:
    print(f"{os.path.basename(path)}: {caption}")

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Time taken: 2.6840641498565674
Time taken: 2.4764938354492188
Time taken: 1.8138954639434814
Time taken: 1.8158586025238037
Time taken: 2.3826143741607666
Time taken: 1.916088342666626
Time taken: 2.21797776222229
Time taken: 1.7319316864013672
Time taken: 2.200808525085449
Time taken: 2.195845365524292
Time taken: 2.191230297088623
Time taken: 2.5566318035125732
Time taken: 1.196281909942627
a car_1 - Copy.png: A car is driving down a street with a large building in the background.
a car_1.png: A silver car is driving down a busy street.
a car_2 - Copy.png: A silver car with red and yellow stripes is parked on the side of the road.
a car_2.png: A silver car with red and yellow stripes is parked on the side of the road.
a car_3.png: A group of men are working on a car.
a car_4.png: A green car is parked in front of a building.
a car_5.png: A red and blue car is driving down a street.
a car_6.png: A white car is parked in a grassy field.
a car_7.png: A blue car is parked next to a yello

## Model 2 - tinyllava/TinyLLaVA-Phi-2-SigLIP-3.1B

- This is a Tiny version of the llava model available on [hugging face](https://huggingface.co/tinyllava/TinyLLaVA-Phi-2-SigLIP-3.1B) supposedly achieving better overall performance against existing 7B models such as LLaVA-1.5 and Qwen-VL.

- This variant does not appear to support 4bit quantization making it worse off than the **llava-hf/llava-1.5-7b-hf** model.

- **NOT VIABLE**: Model takes 3 minutes to process 25 images using an batch_size=2 (larger batch sizes slow down inference due to model scale).

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os, time
from tqdm import tqdm
from generate_model import generate  # official TinyLLaVA helper

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 2
caption_type = "short"  # "short" = 1 sentence, "long" = paragraph
device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# 2. Load TinyLLaVA model
# -------------------------------
hf_path = 'tinyllava/TinyLLaVA-Phi-2-SigLIP-3.1B'

model = AutoModelForCausalLM.from_pretrained(
    hf_path,
    trust_remote_code=True,
    attn_implementation="eager",
    dtype=torch.float16,
    device_map="auto"
).eval()

config = model.config
tokenizer = AutoTokenizer.from_pretrained(
    hf_path,
    use_fast=True,
    model_max_length=config.tokenizer_model_max_length,
    padding_side=config.tokenizer_padding_side
)

# -------------------------------
# 3. Helper to load image paths
# -------------------------------
def load_image_paths(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(exts)]

image_paths = load_image_paths(image_dir)
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Caption generation
# -------------------------------
def generate_captions_batch(paths, caption_type="short"):
    if caption_type == "short":
        prompt_text = "Describe this image in one sentence."
    else:
        prompt_text = "Describe this image in one paragraph."

    captions = []

    for path in paths:
        start_time = time.time()
        # TinyLLaVA generate() accepts local image paths
        output_text, generation_time = generate(
            prompt=prompt_text,
            image=path,
            model=model,
            tokenizer=tokenizer
        )
        print(f"Processed Batch in {generation_time:.2f}s")
        captions.append((path, output_text))

    return captions

# -------------------------------
# 5. Process images in batches
# -------------------------------
all_results = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    batch_results = generate_captions_batch(batch_paths, caption_type=caption_type)
    all_results.extend(batch_results)

# -------------------------------
# 6. Output results
# -------------------------------
print("\n=== Captions ===")
for path, caption in all_results:
    print(f"{os.path.basename(path)}: {caption}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Found 25 images


Processing batches:   0%|          | 0/13 [00:00<?, ?it/s]WARNING:root:inference device is not set, using cuda:0, NVIDIA GeForce RTX 3060 Ti
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Processed Batch in 3.06s


...

Processed Batch in 2.00s


Processing batches:  92%|█████████▏| 12/13 [03:06<00:08,  8.10s/it]WARNING:root:inference device is not set, using cuda:0, NVIDIA GeForce RTX 3060 Ti


Processed Batch in 2.43s


Processing batches: 100%|██████████| 13/13 [03:08<00:00, 14.48s/it]

Processed Batch in 2.20s

=== Captions ===
a car_1 - Copy.png: A silver car is driving down a city street with palm trees and buildings in the background.
a car_1.png: A silver car is driving down a street with people and palm trees in the background.
a car_2 - Copy.png: A boy in a red shirt stands in front of a car with a red stripe on the side.
a car_2.png: A boy in a red shirt is walking on the sidewalk next to a car with a red stripe.
a car_3.png: A man in a suit is helping another man out of a car that has a roof rack on top.
a car_4.png: A green car is parked on the side of a street in front of a building.
a car_5.png: A red, white, and blue car is driving down a street.
a car_6.png: A white car is parked in a grassy field.
a car_7.png: A blue car with a yellow bike attached to it is parked on the street.
a car_8.png: A car with a colorful paint job and a sign that reads "A.R.C.T.I.R.E.M.E.N.G.E.T.A.L.A.T.I.N.G.E.T.A.L.A.T.I.N.G.E.T.A.L.A.T.I.N.G.E.T.A.L.A.T.I.N.G.E.T.A.L.A.T.I.N

## Model 3 - llava-hf/llava-onevision-qwen2-0.5b-si-hf

- This is another llava model variant available on [hugging face](https://huggingface.co/llava-hf/llava-onevision-qwen2-0.5b-si-hf) supposedly achieving better overall performance against existing 7B models such as LLaVA-1.5 and Qwen-VL.

- This variant does further support **flash_attention** to enhance execution time however I didn't manage to get this working.

- The [llava-hf/llava-onevision-qwen2-7b-si-hf](https://huggingface.co/llava-hf/llava-onevision-qwen2-7b-si-hf) model was also tested however being a larger scale model performance was worse.

- **NOT VIABLE**: Model takes 3 minutes to process 25 images using an batch_size=2 (larger batch sizes slow down inference due to model scale).

In [3]:
import torch
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
import os
import time
from torch import inference_mode, autocast
import torch

# ------------------------------- Configuration -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
device = "cuda" if torch.cuda.is_available() else "cpu"
question = "Describe this image in one sentence."
max_new_tokens = 100
batch_size = 2  # adjust depending on GPU memory

# ------------------------------- Load Model -------------------------------
model_id = "llava-hf/llava-onevision-qwen2-0.5b-si-hf" #"llava-hf/llava-onevision-qwen2-7b-si-hf"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"        
)

model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    dtype=torch.float16,
    low_cpu_mem_usage=True
    # use_flash_attention_2=True
).to(device)


model = torch.compile(model, mode="reduce-overhead")

processor = AutoProcessor.from_pretrained(model_id, use_fast=True)

# ------------------------------- Load Images -------------------------------
def load_images_from_folder(folder):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    images = []
    paths = []
    for f in os.listdir(folder):
        if f.lower().endswith(exts):
            path = os.path.join(folder, f)
            try:
                img = Image.open(path).convert("RGB")
                images.append(img)
                paths.append(path)
            except Exception as e:
                print(f"Skipping {path}: {e}")
    return images, paths

images, paths = load_images_from_folder(image_dir)
print(f"Loaded {len(images)} images")

# ------------------------------- Generate Captions in Batches -------------------------------
captions = []
model.eval()
with torch.no_grad():
    for i in range(0, len(images), batch_size):
        batch_images = images[i:i+batch_size]
        batch_paths = paths[i:i+batch_size]

        # Prepare conversation for all images in the batch
        conversation = [{
            "role": "user",
            "content": [{"type": "text", "text": question}] + [{"type": "image"} for _ in batch_images]
        }]
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

        start = time.time()
        # Processor handles multiple images at once
        inputs = processor(images=batch_images, text=prompt, return_tensors="pt").to(model.device, torch.float16)

        with inference_mode():
            with autocast(device_type="cuda", dtype=torch.float16):
                outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        
        # Decode each output
        batch_captions = [processor.decode(out[2:], skip_special_tokens=True) for out in outputs]

        captions.extend(zip(batch_paths, batch_captions))
        print(f"Processed batch {i//batch_size+1} in {time.time()-start:.2f}s")

# ------------------------------- Show results -------------------------------
print("\n=== Captions ===")
for path, cap in captions:
    print(f"{os.path.basename(path)}: {cap}")

Unused or unrecognized kwargs: batch_num_images.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Loaded 25 images


Unused or unrecognized kwargs: batch_num_images.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Processed batch 1 in 33.03s


Unused or unrecognized kwargs: batch_num_images.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Unused or unrecognized kwargs: batch_num_images.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Processed batch 7 in 35.29s


Unused or unrecognized kwargs: batch_num_images.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Processed batch 8 in 35.01s


KeyboardInterrupt: 

## Model 4 - Salesforce/blip2-flan-t5-xl

- The [Salesforce/blip2-flan-t5-xl](https://huggingface.co/Salesforce/blip2-flan-t5-xl) model is an **image captioning model** as opposed to the llava model which is a Multimodal Large Language Model. The focus of this model is to caption images hence it can't generate short vs long captions as it doesn't have the capability to follow instructions.

- This model produces captions similar to the short captions seen in the ["Understanding Bias in Large-Scale Visual Datasets"](https://arxiv.org/pdf/2412.01876v1) paper offering a basic description of the image contents

- The model supports only 8-bit qunatization as applying 4bit quantization by uncommenting the code below results in the model producing gibberish in terms of captions.

- **NOT VIABLE**: Uisng 8-bit quantization isn't sufficient to significantly speed up execution as the model still takes 2.28s to process 25 images using a batch_size=25 (larger batch sizes are supported due to model scale being small).


In [17]:
import os
import time
from PIL import Image
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

# ---------------- Settings ----------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 25
# device is defined but device_map="auto" will handle device assignment
device = "cuda" if torch.cuda.is_available() else "cpu" 
caption_type = "short"  # "short" or "long"

# ---------------- Prompt templates ----------------
short_prompt = "Describe this image in one sentence."
long_prompt = "Describe this image in one paragraph."
custom_prompt = short_prompt if caption_type == "short" else long_prompt

# ---------------- Model setup ----------------
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl", use_fast = True)

# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16 # <--- Requires float16 inputs for computation
# )

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    quantization_config = quantization_config,
    use_safetensors=True,
    device_map="auto"
)

# ---------------- Max tokens ----------------
max_length = 100

# ---------------- Collect image paths ----------------
image_paths = [
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"Found {len(image_paths)} images.")

# ---------------- Loop through images in batches ----------------
start_time = time.time()
total_images = 0

for i in range(0, len(image_paths), batch_size):
    batch_paths = image_paths[i:i+batch_size]
    images = []
    valid_paths = []

    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            images.append(img)
            valid_paths.append(path)
        except Exception as e:
            print(f"Skipping {path}, error: {e}")
            continue

    if not images:
        continue

    # Preprocess batch with instruction prompt (tensors start on CPU)
    inputs = processor(
        images=images,
        text=[custom_prompt] * len(images),
        return_tensors="pt"
    )

    # Move tensors to the appropriate device (GPU) and convert pixel_values to float16
    for k, v in inputs.items():
        if v is not None:
            # Determine the target device (e.g., cuda:0)
            target_device = torch.device(device) 

            # Move tensor to the device
            v = v.to(target_device) 
            
            # If it's a floating-point tensor (the image pixel values), 
            # convert it to float16, as required by bnb_4bit_compute_dtype
            if v.dtype == torch.float32:
                 v = v.to(torch.float16)
            
            inputs[k] = v
            
    # Generate captions
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_length
        )
        captions = [processor.decode(g, skip_special_tokens=True) for g in output_ids]

    # Print results
    for path, caption in zip(valid_paths, captions):
        print(f"{os.path.basename(path)} -> {caption}")

    total_images += len(valid_paths)

end_time = time.time()
elapsed = end_time - start_time
print(f"\nProcessed {total_images} images in {elapsed:.2f} seconds ({total_images/elapsed:.2f} imgs/sec)")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Found 25 images.
a car_1 - Copy.png -> a silver car drives down a street in a city
a car_1.png -> a vintage car drives down a city street
a car_2 - Copy.png -> a car parked on a street
a car_2.png -> a car parked on a street
a car_3.png -> a man is standing next to a car with a hat on
a car_4.png -> a green car parked on a street
a car_5.png -> a red and white car driving down a road
a car_6.png -> a white car parked in a field
a car_7.png -> a blue car with a flaming flame on the side
a car_8.png -> a car with a tiger on the bonnet
a car_9.png -> a man sits on a car seat in front of a car
a girl_1.png -> a girl in a twirls around a frame in a framed frame
a girl_3.png -> a colorful office with a mural on the wall
a girl_4.png -> a pair of people in a room with a mirror
a girl_5.png -> a collage of images of a street scene in a city
a girl_7.png -> a painting of a room with a window
a girl_8.png -> a drawing of a building with a doorway and windows
a_car_0.png -> a car driving down a r

## Model 5 - vikhyatk/moondream2

- **NOT VIABLE**: Model doesn't appear to support batching

In [4]:
# Moondream 2 doesn't seem to support batching

from transformers import AutoModelForCausalLM
from PIL import Image
import torch

# Load the model
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="cuda",
)

# Load your image
image = Image.open(r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\a girl_4.png")

# Optionally set sampling settings
# settings = {"temperature": 0.5, "max_tokens": 768, "top_p": 0.3}
settings = {"temperature": 0.5, "max_tokens": 100, "top_p": 0.3}

# Generate a short caption
short_result = model.caption(
    image, 
    length="short", 
    settings=settings
)
print(short_result)

{'caption': 'Two black and white photographs of two silhouetted individuals are displayed on a beige wall, with the left image slightly higher than the right.'}


## Model 5 - Salesforce/blip-image-captioning-large

- The [Salesforce/blip-image-captioning-large](https://huggingface.co/Salesforce/blip-image-captioning-large) is a predecessor to the [Salesforce/blip2-flan-t5-xl](https://huggingface.co/Salesforce/blip2-flan-t5-xl) model being an **image captioning model** as opposed to the llava model which is a Multimodal Large Language Model. The focus of this model is to caption images hence it can't generate short vs long captions as it doesn't have the capability to follow instructions.

- This model produces more basic captions than its predecessor however the captions are similar to the short captions seen in the ["Understanding Bias in Large-Scale Visual Datasets"](https://arxiv.org/pdf/2412.01876v1) paper offering a basic description of the image contents.

- The current implemented approch does not use quantization as inference speed is sufficient as is.

- **PAPER**: https://arxiv.org/abs/2201.12086

- **VIABLE**: Model takes 2.28s to process 25 images using a batch_size=25 (larger batch sizes are supported due to model scale being small).

In [1]:
import os
import time
from PIL import Image
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

# ---------------- Settings ----------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 50  # Adjust based on GPU memory
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- Prefix template ----------------
output_prefix = "An image of"

# ---------------- Model setup ----------------
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large", use_fast=True)
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large",
    use_safetensors=True
)
model = model.to(device)
model.eval()

# ---------------- Max tokens ----------------
max_length = 100

# ---------------- Collect image paths ----------------
image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))] * 4

print(f"Found {len(image_paths)} images.")

# ---------------- Loop through images in batches ----------------
start_time = time.time()
total_images = 0

for i in range(0, len(image_paths), batch_size):
    batch_paths = image_paths[i:i+batch_size]
    images = []
    valid_paths = []

    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            images.append(img)
            valid_paths.append(path)
        except Exception as e:
            print(f"Skipping {path}, error: {e}")
            continue

    if not images:
        continue

    # Preprocess batch WITHOUT a literal prompt
    inputs = processor(
        images=images, 
        text=[output_prefix]*len(images),  # pass the instruction
        return_tensors="pt"
    ).to(device, torch.float16)

    # Generate captions (length controlled by max_length)
    with torch.no_grad():
        out = model.generate(**inputs, 
                             max_new_tokens=max_length, 
                             num_beams=1, 
                             do_sample=True,          # Optional: Increase inference time for more creative outputs
                             top_k=50,                # Optional: Control word choices
                             temperature=0.7          # Optional: Control randomness (0.7-0.9 is good)
                            )
        captions = processor.batch_decode(out, skip_special_tokens=True)

    # Print results
    for path, caption in zip(valid_paths, captions):
        print(f"{os.path.basename(path)} -> {caption}")

    total_images += len(valid_paths)

elapsed = time.time() - start_time
print(f"\nProcessed {total_images} images in {elapsed:.2f} seconds ({total_images/elapsed:.2f} imgs/sec)")

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Found 100 images.
a car_1 - Copy.png -> an image of a silver van is parked next to a tall building
a car_1.png -> an image of a car driving down the road with many people
a car_2 - Copy.png -> an image of a car that is parked in front of a building
a car_2.png -> an image of a silver car traveling down a street next to a tree
a car_3.png -> an image of two people fixing a car on the side of the road
a car_4.png -> an image of a green car is parked on the side of the road
a car_5.png -> an image of a car painted in red, white and blue paint
a car_6.png -> an image of a white, classic american car sitting in a field
a car_7.png -> an image of a blue car that is painting on the side of a road
a car_8.png -> an image of a yellow car with a surfboard on the front of it
a car_9.png -> an image of a man leaning on a car parked on a street
a girl_1.png -> an image of a girl is standing in a mirror with her arms out
a girl_3.png -> an image of a long hallway in a building with a couple of chair

## Model 6 - microsoft/Florence-2-base-ft

- The [microsoft/Florence-2-base-ft](https://huggingface.co/microsoft/Florence-2-base-ft) is a vision foundation model that uses a prompt-based approach to handle a wide range of vision and vision-language tasks. Florence-2 can interpret simple text prompts to perform tasks like captioning, object detection, and segmentation.

The larger [microsoft/Florence-2-base-ft](https://huggingface.co/microsoft/Florence-2-base-ft) was also tested with 4-bit quantization howeve it takes 7 sec for 25 images

- This model supports the concept of **\<CAPTION\>** & **\<DETAILED_CAPTION\>** which outline the lenght and detail the model should provide in its captions similar to the the short vs long captions in the  ["Understanding Bias in Large-Scale Visual Datasets"](https://arxiv.org/pdf/2412.01876v1) paper.

- Their exists two size varients the base & large however the latter is much more intensive and takes to long to carry out inference on 25 images.

- The current implemented approch does not use quantization as inference speed is sufficient as is.

- **PAPER**: https://arxiv.org/abs/2311.06242

- **VIABLE**: Model takes 2.55 to process 25 images using a batch_size=25 (larger batch sizes are supported due to model scale being small).

In [9]:
# VIABLE MODEL PROCESSES 200 IMAGES IN 24 SEC
import os
os.environ["ATTN_IMPLEMENTATION"] = "eager"
os.environ["TRANSFORMERS_NO_SDPA"] = "1"

import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from PIL import Image
import numpy as np
from tqdm import tqdm
import time

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 25                # Adjust for GPU VRAM (Florence-2 is heavy)
max_new_tokens = 100           # Length of captions
prompt_text = "<DETAILED_CAPTION>" #"<CAPTION>" 

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

np.float_ = np.float64
np.complex_ = np.complex128

# # Quantization config (4-bit)
# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )

# -------------------------------
# 2. Load model and processor
# -------------------------------
model_name = "microsoft/Florence-2-base-ft" #"microsoft/Florence-2-large-ft"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch_dtype,
    trust_remote_code=True,
    attn_implementation="eager",
    # quantization_config=quant_config
).to(device).eval()

processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

# -------------------------------
# 3. Helper to load images
# -------------------------------
def load_images_from_dir(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(exts)]

image_paths = load_images_from_dir(image_dir) * 8
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Batched inference
# -------------------------------
all_captions = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    images = [Image.open(p).convert("RGB") for p in batch_paths]

    inputs = processor(
        text=[prompt_text] * len(images),
        images=images,
        return_tensors="pt",
        padding=True
    ).to(device, torch_dtype)

    start_time = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,       # Beam search off
            use_cache=False    # Disable caching - otherwise crashes the model
        )

    decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)

    for path, text, img in zip(batch_paths, decoded, images):
        caption = processor.post_process_generation(
            text,
            task=prompt_text,
            image_size=(img.width, img.height)
        )
        all_captions.append((os.path.basename(path), caption))

    print(f"Batch {i//batch_size + 1}: {len(batch_paths)} images processed in {time.time() - start_time:.2f}s")
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -------------------------------
# 5. Output results
# -------------------------------
print("\n=== Captions ===")
for img_name, caption in all_captions:
    print(f"{img_name}: {caption[prompt_text]}")


Found 200 images


Processing batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batch 1: 25 images processed in 12.37s


Processing batches:  12%|█▎        | 1/8 [00:13<01:34, 13.52s/it]

Batch 2: 25 images processed in 6.91s


...

Batch 7: 25 images processed in 6.90s


Processing batches: 100%|██████████| 8/8 [01:09<00:00,  8.16s/it]

Batch 8: 25 images processed in 6.90s


Processing batches: 100%|██████████| 8/8 [01:09<00:00,  8.70s/it]


=== Captions ===
a car_1 - Copy.png: In this image we can see a car on the road. In the background we can also see a group of people, trees, buildings, poles, sky and also the water.
a car_1.png: In this image we can see a car on the road. In the background we can also see a group of people, trees, buildings, street lights, poles, boards and sky.
a car_2 - Copy.png: In this image I can see a car on the road. There is a person standing on the footpath. There are trees and buildings at the back.
a car_2.png: In this image I can see a car on the road. There is a person standing on the footpath. There are trees and buildings at the back.
a car_3.png: In this image I can see two persons are standing on the road. I can also see a car. In the background I can observe a fence and some trees.
a car_4.png: In this image I can see a green color car on the road. In the background I can observe a building. There are some trees on either sides of the road and there are some people walking on the le

## Model 7 - microsoft/kosmos-2-patch14-224

- The [microsoft/kosmos-2-patch14-224](https://huggingface.co/microsoft/kosmos-2-patch14-224) model is a Grounding Multimodal Large Language Models.

- This model is capable of performing [different tasks](https://huggingface.co/microsoft/kosmos-2-patch14-224#:~:text=Here%20are%20the%20tasks%20Kosmos%2D2%20could%20perform) through changing the prompts:
    - Phrase Grounding
    - Referring Expression Comprehension
    - Referring expression generation
    - Grounded VQA
    - Grounded VQA with multimodal referring via bounding boxes
    - Brief
    - Detailed

- Model supports short vs long prompts via the prompts **"\<grounding\> An image of"** and **"\<grounding\> Describe this image in detail:"** respectively.

- **PAPER**: https://arxiv.org/abs/2306.14824

- **VIABLE**: Model takes 15s to process 200 images using a batch_size=200.

In [3]:
# VIABLE MODEL PROCESSES 200 IMAGES IN 15 SEC
import torch
from transformers import AutoProcessor, Kosmos2ForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
import os, time
from tqdm import tqdm

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 200                # Adjust for GPU VRAM (try 2–8 for 8GB GPU)
max_new_tokens = 100                # Lower = faster; higher = more descriptive captions

# Quantization config (4-bit)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# -------------------------------
# 2. Load model and processor
# -------------------------------
model_name = "microsoft/kosmos-2-patch14-224"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Kosmos2ForConditionalGeneration.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quant_config
).eval()

processor = AutoProcessor.from_pretrained(model_name, use_fast=True)

# -------------------------------
# 3. Helper to load images
# -------------------------------
def load_images_from_dir(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    image_files = [os.path.join(directory, f) for f in os.listdir(directory)
                   if f.lower().endswith(exts)]
    return image_files

image_paths = load_images_from_dir(image_dir) * 40
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Batched inference
# -------------------------------
all_captions = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    images = [Image.open(p).convert("RGB") for p in batch_paths]

    prompt = "<grounding> Describe this image in detail:" # "<grounding> An image of"
    inputs = processor(text=[prompt] * len(images), images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            pixel_values=inputs["pixel_values"],
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            image_embeds_position_mask=inputs["image_embeds_position_mask"],
            max_new_tokens=max_new_tokens,
            use_cache=True
        )

    decoded = processor.batch_decode(outputs, skip_special_tokens=True)
    for path, text in zip(batch_paths, decoded):
        caption, _ = processor.post_process_generation(text)
        all_captions.append((os.path.basename(path), caption))

    print(f"Batch {i//batch_size + 1}: {len(batch_paths)} images processed in {time.time() - start_time:.2f}s")
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -------------------------------
# 5. Output results
# -------------------------------
print("\n=== Captions ===")
for img_name, caption in all_captions:
    print(f"{img_name}: {caption}")


Found 1000 images


Processing batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batch 1: 200 images processed in 17.26s


Processing batches:  20%|██        | 1/5 [00:20<01:23, 20.82s/it]

Batch 2: 200 images processed in 16.26s


...

Batch 4: 200 images processed in 15.61s


Processing batches:  80%|████████  | 4/5 [01:20<00:19, 19.90s/it]

Batch 5: 200 images processed in 17.50s


Processing batches: 100%|██████████| 5/5 [01:41<00:00, 20.33s/it]


=== Captions ===
a car_1 - Copy.png: Describe this image in detail: A large group of people are standing on the side of the road, with a large group standing in front of a building. There are also a few cars parked on the street, and a few people are walking around. The scene is set in a city, with the tall buildings in the
a car_1.png: Describe this image in detail: A car is parked in front of a building with palm trees and a blue sky.
a car_2 - Copy.png: Describe this image in detail: The image features a man standing in front of a building, with a car parked in front. The scene is set in a parking lot, with the man standing next to a car. There are two cars parked in the lot, one on the left side and the other on the right side of the scene
a car_2.png: Describe this image in detail: The image features a man standing in front of a building, with a car parked in front. The scene is set in a parking lot, with the man standing next to a car. There are two cars parked in the lot, one o

## Model 8 - nlpconnect/vit-gpt2-image-captioning

- The [nlpconnect/vit-gpt2-image-captioning](https://huggingface.co/nlpconnect/vit-gpt2-image-captioning) model is an image captioning model.

- **PAPER**: https://ankur3107.github.io/blogs/the-illustrated-image-captioning-using-transformers/

- **VIABLE**: Model takes 4s to process 200 images using a batch_size=25, however the captions are very basic and not as detaild in comparison to other small scale models.

In [1]:
import torch
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import os, time
from tqdm import tqdm

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 25                 # Adjust to GPU VRAM (8GB → small batches)
max_length = 100                 # Caption length

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# 2. Load model and processor
# -------------------------------
model_name = "nlpconnect/vit-gpt2-image-captioning"
model = VisionEncoderDecoderModel.from_pretrained(model_name).to(device).eval()
processor = ViTImageProcessor.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# -------------------------------
# 3. Helper to load images
# -------------------------------
def load_images_from_dir(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(exts)]

image_paths = load_images_from_dir(image_dir) * 8
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Batched inference
# -------------------------------
all_captions = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    images = [Image.open(p).convert("RGB") for p in batch_paths]

    # Preprocess images
    pixel_values = processor(images=images, return_tensors="pt").pixel_values.to(device)

    start_time = time.time()
    with torch.no_grad():
        output_ids = model.generate(pixel_values, max_length=max_length, num_beams=3)

    captions = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    for path, caption in zip(batch_paths, captions):
        all_captions.append((os.path.basename(path), caption))

    print(f"Batch {i//batch_size + 1}: {len(batch_paths)} images processed in {time.time() - start_time:.2f}s")
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -------------------------------
# 5. Output results
# -------------------------------
print("\n=== Captions ===")
for img_name, caption in all_captions:
    print(f"{img_name}: {caption}")


Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


Found 200 images


Processing batches:   0%|          | 0/8 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.
Processing batches:  12%|█▎        | 1/8 [00:01<00:10,  1.43s/it]

Batch 1: 25 images processed in 0.97s


Processing batches:  25%|██▌       | 2/8 [00:02<00:07,  1.24s/it]

...

Batch 6: 25 images processed in 0.65s


Processing batches:  88%|████████▊ | 7/8 [00:07<00:01,  1.10s/it]

Batch 7: 25 images processed in 0.65s


Processing batches: 100%|██████████| 8/8 [00:09<00:00,  1.13s/it]

Batch 8: 25 images processed in 0.65s

=== Captions ===
a car_1 - Copy.png: a street view of a train going down the tracks 
a car_1.png: a car driving down a street next to tall buildings 
a car_2 - Copy.png: a car is parked on the side of the road 
a car_2.png: a car is parked on the side of the road 
a car_3.png: a man and a woman are standing in front of a truck 
a car_4.png: a green car is parked in front of a building 
a car_5.png: a red and white car parked in front of a building 
a car_6.png: a white car parked in a grassy field 
a car_7.png: a car is parked next to a bicycle on the street 
a car_8.png: a vintage car with a picture of a man on it 
a car_9.png: a white car is parked in front of a building 
a girl_1.png: a woman standing in front of a mirror in a room 
a girl_3.png: a painting of a mannequin on a wall 
a girl_4.png: two pictures of people standing in front of a wall 
a girl_5.png: a collage of photos of a living room with furniture 
a girl_7.png: a painting of a w

## Model 9 - cnmoro/tiny-image-captioning

- The [cnmoro/tiny-image-captioning](https://huggingface.co/cnmoro/tiny-image-captioning) model is an image captioning model, based on bert-tiny and vit-small, weighing only 100mb. 

- This model is extremly small scale and only being considered for the worst case scenario where no other models are viable.

- **PAPER**: No Paper Available

- **VIABLE**: Model takes 1s to process 200 images using a batch_size=200, however the captions are very basic and appear to be not as accurate to model content as other valid models.

In [2]:
import torch
from transformers import VisionEncoderDecoderModel, AutoTokenizer, AutoImageProcessor
from PIL import Image
import os, time
from tqdm import tqdm

# -------------------------------
# 1. Configuration
# -------------------------------
image_dir = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images"
batch_size = 200
device = "cuda" if torch.cuda.is_available() else "cpu"
max_length = 64

# -------------------------------
# 2. Load model, tokenizer, and processor
# -------------------------------
model_name = "cnmoro/tiny-image-captioning"
model = VisionEncoderDecoderModel.from_pretrained(model_name).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)
image_processor = AutoImageProcessor.from_pretrained(model_name)

# -------------------------------
# 3. Helper to load images
# -------------------------------
def load_images_from_dir(directory):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.lower().endswith(exts)]

image_paths = load_images_from_dir(image_dir) * 8
print(f"Found {len(image_paths)} images")

# -------------------------------
# 4. Batched inference
# -------------------------------
all_captions = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing batches"):
    batch_paths = image_paths[i:i + batch_size]
    images = [Image.open(p).convert("RGB") for p in batch_paths]

    # Preprocess images
    pixel_values = image_processor(images, return_tensors="pt").pixel_values.to(device)

    start_time = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=max_length,
            num_beams=3,      # 1 for faster but slightly lower quality
            temperature=0.7,
            top_p=0.8,
            top_k=50
        )

    captions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    for path, caption in zip(batch_paths, captions):
        all_captions.append((os.path.basename(path), caption))

    print(f"Batch {i//batch_size + 1}: {len(batch_paths)} images processed in {time.time() - start_time:.2f}s")
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# -------------------------------
# 5. Output results
# -------------------------------
print("\n=== Captions ===")
for img_name, caption in all_captions:
    print(f"{img_name}: {caption}")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Found 200 images


Processing batches:   0%|          | 0/1 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=25) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
Processing batches: 100%|██████████| 1/1 [00:04<00:00,  4.50s/it]

Batch 1: 200 images processed in 0.81s

=== Captions ===
a car_1 - Copy.png: a man is sitting on the sidewalk.
a car_1.png: a man is walking through the street.
a car_2 - Copy.png: a man is walking through the street.
a car_2.png: a man is walking through the street.
a car_3.png: a man is sitting on the sidewalk.
a car_4.png: a man is walking through the street.
a car_5.png: a yellow car is running through the water.
a car_6.png: a man is walking through a wooded area.
a car_7.png: a man is sitting on the street.
a car_8.png: a yellow car is stopped in front of a car.
a car_9.png: a man is sitting on the street.
a girl_1.png: a woman is sitting in front of a building.
a girl_3.png: a man wearing a blue shirt is sitting on the floor.
a girl_4.png: a group of people are in front of a building.
a girl_5.png: a man is sitting on a bench.
a girl_7.png: a man is sitting in front of a building.
a girl_8.png: a man is sitting on a bench.
a_car_0.png: a man is walking through a street.
a_girl_0

# Testing VIABLE Models 

- This test will determine the best model in terms of speed, output quality and additional functionality (were applicable).

- Testing will be carried out on the **XXX** Dataset using the procedure outlined in **XXX** via the **XXX** metric

### Relevant Papers:

- [Improving Image Captioning Descriptiveness by Ranking and LLM-based Fusion](https://arxiv.org/html/2306.11593)